# CSCI/MATH 485 Assignment
## Customer Churn Prediction with XGBoost
## Starter Notebook

This notebook is compatible with **Jupyter Notebook** and **Google Colab**.

This starter code is only to get you started. You can change any of the existing code here to complete all the tasks.

Complete all `TODO` sections. Make sure your final submission includes:
- data analysis,
- a tuned XGBoost model,
- your chosen main evaluation metric and justification,
- interpretation of top important features,
- and a final comparison with the baseline model.


## 1. Setup


In [29]:
# If you are using Google Colab, uncomment the next line if xgboost is not installed.
# !pip install xgboost


In [52]:
import sys
print(sys.executable)

/opt/anaconda3/bin/python


In [53]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier


## 2. Load the Dataset

**Instructions for students**
- Load the IBM Telco Customer Churn dataset.
- Display the first few rows.
- Confirm the dataset shape.
- If the url doesn't work for you, download the csv file from the Canvas Assignment#4

In [54]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/telco-customer-churn-by-IBM.csv"
df = pd.read_csv(url)

# Display the first 5 rows of the dataset (done below)
df.head()

# TODO:
# Display the shape of the dataset
# Number of rows and columns
print("Shape:", df.shape)
print("Num rows:", df.shape[0])
print("Num columns:", df.shape[1])




Shape: (7043, 21)
Num rows: 7043
Num columns: 21


## 3. Data Exploration

Complete the following:
- Print all column names
- Show data types
- Count missing values in each column
- Show the distribution of the target variable
- Write a short note: Is this a classification or regression problem? Why is this useful in business?


In [22]:
# TODO:
# Print column names
# Print data types
# Print missing values for each column
# Print value counts for the target column

print("Columns:")
# your code here
print(df.columns.tolist())



print("\nData types:")
# your code here
print(df.dtypes)


print("\nMissing values:")
# your code here
print(df.isna().sum())


print("\nTarget distribution:")
# your code here
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True))

# Extra: display other information of the dataset that you think can be useful
print("\nDataset shape:")
print(df.shape)

print("\nBlank strings per column:")
print((df == " ").sum())



Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Missing values:
customerID          0
gender       

**TODO (write your answer below):**

1. What kind of machine learning problem is this?
2. Why is churn prediction important in a business setting?

_Replace this text with your answer._


## 4. Basic Cleaning

Complet the following:
- Identify whether there is an ID column that should be removed
- Convert the target column into binary form if needed
- Convert any numeric-looking columns stored as text into numeric values


In [23]:
# Make a copy so the original data remains unchanged
df_clean = df.copy()

# TODO:
# 1. Drop any unnecessary identifier column(s)
df_clean = df_clean.drop(columns=["customerID"])
# 2. Convert the target column to 0/1
df_clean["Churn"] = df_clean["Churn"].map({"Yes": 1, "No": 0})
# 3. Convert any columns that should be numeric into numeric type
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

# 4. Handle invalid parsing if needed

# Example structure only:
# df_clean = df_clean.drop(columns=[...])
# df_clean["Churn"] = df_clean["Churn"].map({"Yes": 1, "No": 0})
# df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

print("Updated data types:")
print(df_clean.dtypes)

print("\nMissing values after cleaning:")
print(df_clean.isna().sum())

df_clean.head()


Updated data types:
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

Missing values after cleaning:
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
Paymen

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 5. Define Features and Target

Complet the following:
- Define `X` and `y`
- Set the correct target column


In [33]:
# TODO:
# Replace with the correct target column name
target_col = "Churn"

X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


Feature matrix shape: (7043, 19)
Target shape: (7043,)


## 6. Identify Numeric and Categorical Features

Complet the following:
- Create a list of numeric columns
- Create a list of categorical columns


In [25]:
# TODO:
# Identify numeric and categorical feature columns

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


Numeric features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 7. Train/Test Split

Complet the following:
- Split the dataset into training and testing sets
- Use stratification if appropriate


In [36]:
# TODO:
# Create train/test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


X_train shape: (5634, 19)
X_test shape: (1409, 19)
y_train shape: (5634,)
y_test shape: (1409,)


## 8. Preprocessing Pipelines

Build preprocessing for:
- numeric features
- categorical features

Then combine them into a `ColumnTransformer`.


In [28]:
# TODO:
# Build numeric preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    # Example:
    ("imputer", SimpleImputer(strategy="median"))
])

# TODO:
# Build categorical preprocessing pipeline
categorical_transformer = Pipeline(steps=[
    # Example:
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# TODO:
# Combine both preprocessing pipelines
preprocessor = ColumnTransformer(transformers=[
    # Example:
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod'])])

## 9. Baseline Model: Logistic Regression

Complet the following:
- Train a Logistic Regression model as the baseline
- Generate predictions
- Evaluate using multiple metrics
- You may need to adjust `max_iter` if the model is not converging.


In [55]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=5000))
])

# TODO:
# Fit the baseline model
# Generate predicted labels
# Generate predicted probabilities

# your code here
baseline_model.fit(X_train, y_train)
baseline_preds=baseline_model.predict(X_test)
baseline_probs=baseline_model.predict_proba(X_test)[:, 1]


In [57]:
# TODO:
# Compute and print:
# - Accuracy
# - Precision
# - Recall
# - F1-score
# - ROC-AUC

# Suggested variable names:
# baseline_preds
# baseline_probs

# your code here
print("accuracy:", accuracy_score(y_test, baseline_preds))
print("precision:", precision_score(y_test, baseline_preds))
print("recall:", recall_score(y_test, baseline_preds))
print("f1-score:", f1_score(y_test, baseline_preds))
print("ROC-AUC:", roc_auc_score(y_test, baseline_probs))


accuracy: 0.8076650106458482
precision: 0.6614420062695925
recall: 0.5641711229946524
f1-score: 0.6089466089466089
ROC-AUC: 0.8420729029424682


In [39]:
# Optional but helpful
# TODO:
# Print classification report and confusion matrix

# your code here

print("\nClassification Report:")
print(classification_report(y_test, baseline_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, baseline_preds))


Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1035
           1       0.66      0.56      0.61       374

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409


Confusion Matrix:
[[927 108]
 [163 211]]


## 10. Choose Your Main Evaluation Metric

Choose **one main metric** for this churn problem.

You must explain:
- which metric you chose,
- why it is appropriate,
- and why it is more informative than accuracy alone for this problem.


**TODO (write your answer below):**

_Replace this text with your answer._


I chose F1 score as the main evaluation metric for this churn prediction problem. F1 score is a good choice because churn prediction needs a balance between precision and recall. If precision is too low, the company may spend time and money on customers who were not actually likely to leave. If recall is too low, the model may miss many customers who really are at risk of churning. F1 score is also more useful than accuracy alone for this dataset because the classes are somewhat imbalanced. Most customers do not churn, while a smaller group does. In that kind of situation, a model can seem accurate just by predicting the larger class most of the time. F1 score gives a better picture of how well the model finds real churners without creating too many false alarms.

## 11. XGBoost Model

Complet the following:
- Build an XGBoost pipeline
- Tune at least 3 hyperparameters
- Use either `GridSearchCV` or your own tuning approach


In [40]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ))
])

# TODO:
# Define a hyperparameter grid with at least 3 hyperparameters
param_grid = {
    # Example format:
   "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.05, 0.1]
}

param_grid


{'classifier__n_estimators': [50, 100],
 'classifier__max_depth': [3, 5],
 'classifier__learning_rate': [0.05, 0.1]}

In [41]:
# TODO:
# Run GridSearchCV (or perform manual tuning)
# Choose a scoring metric that matches your selected main evaluation metric

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring=None,   # TODO: replace with your chosen scoring metric
    cv=3,
    n_jobs=-1,
    verbose=1
)

# TODO:
# Fit grid search on training data

# your code here
grid_search.fit(X_train, y_train)


Fitting 3 folds for each of 8 candidates, totalling 24 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median'))]),
                                                                         ['SeniorCitizen',
                                                                          'tenure',
                                                                          'MonthlyCharges',
                                                                          'TotalCharges']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         [...
                                                      max_cat_to_onehot=None,
                                                      max_delta_step=None,
                                                      max_depth=None,
                                                      max_leaves=None,
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=None,
                                                      num_parallel_tree=None, ...))]),
             n_jobs=-1,
             param_grid={'classifier__learning_rate': [0.05, 0.1],
                         'classifier__max_depth': [3, 5],
                         'classifier__n_estimators': [50, 100]},
             verbose=1)

In [ ]:
# TODO:
# Print the best hyperparameters
# Save the best model
# your code here
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

best_xgb_model = grid_search.best_estimator_


Best parameters: {'classifier__learning_rate': 0.1, 'classifier__max_depth': 5, 'classifier__n_estimators': 50}
Best cross-validation score: 0.8074192403265886


## 12. Evaluate the Tuned XGBoost Model

Evaluate XGBoost using the same metrics as the baseline.


In [ ]:
# TODO:
# Generate predictions and probabilities using the best XGBoost model

# Suggested variable names:
# xgb_preds
# xgb_probs

# your code here
xgb_best_model = grid_search.best_estimator_
xgb_preds = xgb_best_model.predict(X_test)
xgb_probs = xgb_best_model.predict_proba(X_test)[:, 1]


In [56]:
# TODO:
# Compute and print:
# - Accuracy
# - Precision
# - Recall
# - F1-score
# - ROC-AUC

# your code here
print("accuracy:", accuracy_score(y_test, xgb_preds))
print("precision:", precision_score(y_test, xgb_preds))
print("recall:", recall_score(y_test, xgb_preds))
print("F1 score:", f1_score(y_test, xgb_preds))
print("ROC-AUC:", roc_auc_score(y_test, xgb_probs))


accuracy: 0.7977288857345636
precision: 0.6459016393442623
recall: 0.5267379679144385
F1 score: 0.5802650957290133
ROC-AUC: 0.844193856725826


In [45]:
# Optional but helpful
# TODO:
# Print classification report and confusion matrix

# your code here
print("\nClassification Report:")
print(classification_report(y_test, xgb_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_preds))



Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.53      0.58       374

    accuracy                           0.80      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409


Confusion Matrix:
[[927 108]
 [177 197]]


## 13. Feature Importance

Use the trained XGBoost model to:
- extract feature importances,
- recover transformed feature names,
- display the top 5 to 10 most important features.


In [46]:
# TODO:
# Access the fitted preprocessor and fitted XGBoost classifier from the pipeline

# Example structure:
# fitted_preprocessor = best_model.named_steps["preprocessor"]
# fitted_xgb = best_model.named_steps["classifier"]

# your code here
fitted_preprocessor = xgb_best_model.named_steps["preprocessor"]
fitted_xgb = xgb_best_model.named_steps["classifier"]


In [47]:
# TODO:
# Get transformed feature names
# Get feature importances
# Create a DataFrame sorted by importance

# Example structure:
# feature_names = fitted_preprocessor.get_feature_names_out()
# importances = fitted_xgb.feature_importances_

# your code here
feature_names = fitted_preprocessor.get_feature_names_out()
importances = fitted_xgb.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)


In [48]:
# TODO:
# Display the top 10 most important features

# your code here
print(feature_importance_df.head(10))

                             Feature  Importance
36      cat__Contract_Month-to-month    0.588191
16  cat__InternetService_Fiber optic    0.117495
18            cat__OnlineSecurity_No    0.030042
15          cat__InternetService_DSL    0.027363
27               cat__TechSupport_No    0.019343
1                        num__tenure    0.016902
37            cat__Contract_One year    0.016894
35          cat__StreamingMovies_Yes    0.015283
38            cat__Contract_Two year    0.014108
12             cat__MultipleLines_No    0.010872


## 14. Interpret the Top Features

Write a short interpretation of the most important features.

Your explanation should:
- use plain language,
- connect features to churn behavior,
- and explain what the company might learn from them.


The most important features suggest that customer commitment, service type, and support options are closely related to churn. The strongest feature was month to month contract, which suggests that customers on shorter contracts are more likely to leave. Customers with one year or two year contracts seemed more stable, so this suggests that longer contracts may help reduce churn.  Fiber optic internet service was also one of the most important features. This suggests that internet service type is related to churn, and the company may want to look more closely at whether fiber optic customers are less satisfied with price, reliability, or overall service.  The features OnlineSecurity_No and TechSupport_No also mattered, which suggests that customers without extra protection or support may be more likely to churn. Tenure was another important feature, showing that how long a customer has stayed with the company also plays a role. Overall, these results suggest that customers on flexible contracts, customers without support related services, and newer customers may be at higher risk of leaving.

**TODO (write your answer below):**

_Replace this text with your answer._


## 15. Final Comparison: Logistic Regression vs XGBoost

Compare the two models using your results.

Your discussion should answer:
- Which model performed better?
- On which metric(s)?
- Why might XGBoost perform better on this dataset?
- What is one limitation of XGBoost?


In [49]:
# TODO:
# Print a side-by-side comparison of the main metrics
# for Logistic Regression and XGBoost

# your code here
comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Logistic Regression": [
        accuracy_score(y_test, baseline_preds),
        precision_score(y_test, baseline_preds),
        recall_score(y_test, baseline_preds),
        f1_score(y_test, baseline_preds),
        roc_auc_score(y_test, baseline_probs)
    ],
    "XGBoost": [
        accuracy_score(y_test, xgb_preds),
        precision_score(y_test, xgb_preds),
        recall_score(y_test, xgb_preds),
        f1_score(y_test, xgb_preds),
        roc_auc_score(y_test, xgb_probs)
    ]
})

print(comparison_df)


      Metric  Logistic Regression   XGBoost
0   Accuracy             0.807665  0.797729
1  Precision             0.661442  0.645902
2     Recall             0.564171  0.526738
3   F1-score             0.608947  0.580265
4    ROC-AUC             0.842073  0.844194


**TODO (write your answer below):**

_Replace this text with your answer._


Based on my results, Logistic Regression performed better overall on this dataset. It had higher accuracy, precision, recall, and F1 score than the tuned XGBoost model. Since I chose F1 score as my main evaluation metric, Logistic Regression was the better model for this assignment. The tuned XGBoost model did slightly better only on ROC AUC. This suggests that XGBoost was a little better at ranking customers by churn risk, but it did not do better than Logistic Regression on the main metric I chose. XGBoost may still work well for this kind of problem because it can capture more complex patterns and interactions between features. One limitation is that it is more complex, harder to interpret, and usually takes more time to tune and train than Logistic Regression.

## 16. Suggested Submission Checklist

Before submitting, make sure your notebook includes:
- completed code cells,
- outputs for each section,
- your selected main evaluation metric and justification,
- tuned XGBoost hyperparameters,
- feature importance results,
- interpretation of important features,
- and a final comparison of the two models.
